# 14 - Review Response Fixes

This notebook contains additional analyses and fixes requested during the peer-review process:

1. **Fixed-model cross-cohort transfer**: Train on TCGA, validate on GSE96058 without refitting.
2. **Binary classification n breakdown**: Detailed accounting of usable samples vs. censored cases.
3. **DeLong significance test**: Statistical comparison between mean z-score and ssGSEA AUCs.
4. **Hazard ratios from combined Cox model**: Reporting HRs and 95% CIs for final model coefficients.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import bootstrap, norm
from lifelines import CoxPHFitter
import os

from src.data_loader import load_tcga_feature_matrix, load_clinical_data
from src.preprocessing import encode_clinical_features, filter_outcome
from src.features import build_feature_matrix

### Section 1: Fixed-model cross-cohort transfer (TRUE external validation)

Train classifiers on ALL TCGA pathway-only features, apply once to GSE96058 without refitting.

In [ ]:
# Load TCGA
tcga = pd.read_csv('../data/processed/02_tcga_feature_matrix.csv')
pathway_cols = ['Pathway_Proliferation','Pathway_Estrogen_Response','Pathway_Immune_Response',
                'Pathway_Invasion_EMT','Pathway_Apoptosis','Pathway_HER2_Signaling',
                'Pathway_Angiogenesis','Ratio_Prolif_Apop']
X_tcga = tcga[pathway_cols].values
y_tcga = tcga['high_risk'].values
mask_tcga = ~np.isnan(y_tcga)
X_tcga, y_tcga = X_tcga[mask_tcga], y_tcga[mask_tcga]

# Load GSE96058 pathway scores (generated in notebook 04)
gse_scores_path = '../data/processed/gse96058_pathway_scores.csv'
if os.path.exists(gse_scores_path):
    gse = pd.read_csv(gse_scores_path)
    # Binary endpoint: high_risk known (not censored before 60mo)
    gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
    gse_binary = gse.merge(gse_clin[['sample_id', 'high_risk']], on='sample_id')
    gse_binary = gse_binary[gse_binary['high_risk'].notna()].copy()
    
    X_gse = gse_binary[pathway_cols].values
    y_gse = gse_binary['high_risk'].values

    # Fit scaler on TCGA only
    scaler = StandardScaler()
    X_tcga_scaled = scaler.fit_transform(X_tcga)
    X_gse_scaled = scaler.transform(X_gse)  # Apply TCGA scaler to GSE

    classifiers = {
        'Elastic Net': LogisticRegression(penalty='elasticnet', solver='saga',
                                           l1_ratio=0.5, C=1.0, max_iter=5000,
                                           class_weight='balanced', random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=500, max_depth=8,
                                                 class_weight='balanced', random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=500,
                                                         learning_rate=0.03,
                                                         max_depth=3, random_state=42)
    }

    results = []
    for name, clf in classifiers.items():
        clf.fit(X_tcga_scaled, y_tcga)
        proba = clf.predict_proba(X_gse_scaled)[:,1]
        auc = roc_auc_score(y_gse, proba)
        # Bootstrap CI
        def auc_stat(y, p): return roc_auc_score(y, p)
        ci = bootstrap((y_gse, proba), auc_stat, n_resamples=1000,
                       paired=True, random_state=42, method='percentile')
        results.append({'Classifier': name, 'AUC': round(auc,3),
                        'CI_lower': round(ci.confidence_interval.low,3),
                        'CI_upper': round(ci.confidence_interval.high,3)})
        print(f"{name}: AUC={auc:.3f} [{ci.confidence_interval.low:.3f}--{ci.confidence_interval.high:.3f}]")

    pd.DataFrame(results).to_csv('../results/transfer_results.csv', index=False)
    print("\nSaved results to results/transfer_results.csv")
else:
    print(f"Warning: {gse_scores_path} not found. Skipping Section 1.")

### Section 2: Binary classification n after censoring

In [ ]:
gse_clinical = pd.read_csv('../data/clinical/01_gse96058_clinical.csv')
total_n = len(gse_clinical)
binary_n = gse_clinical['high_risk'].notna().sum()
high_risk_n = (gse_clinical['high_risk'] == 1).sum()
low_risk_n = (gse_clinical['high_risk'] == 0).sum()
censored_n = total_n - binary_n

print(f"Total GSE96058 n: {total_n}")
print(f"Binary classification n (high_risk known): {binary_n}")
print(f"  High-risk (died <=60mo): {high_risk_n}")
print(f"  Low-risk (alive >=60mo): {low_risk_n}")
print(f"Excluded (censored <60mo): {censored_n}")

# Save
pd.DataFrame([{'total_n': total_n, 'binary_n': binary_n,
               'high_risk_n': high_risk_n, 'low_risk_n': low_risk_n,
               'censored_excluded': censored_n}]).to_csv('../results/binary_n_breakdown.csv', index=False)
print("\nSaved results to results/binary_n_breakdown.csv")

### Section 3: DeLong significance test — mean z-score vs ssGSEA

In [ ]:
def delong_test(auc1, auc2, n_pos, n_neg):
    """Simplified DeLong test for paired AUCs."""
    # Using Hanley-McNeil approximation for correlated AUCs
    q1 = auc1 / (2 - auc1)
    q2 = 2 * auc1**2 / (1 + auc1)
    var1 = (auc1*(1-auc1) + (n_pos-1)*(q1-auc1**2) + (n_neg-1)*(q2-auc1**2)) / (n_pos*n_neg)
    q1b = auc2 / (2 - auc2)
    q2b = 2 * auc2**2 / (1 + auc2)
    var2 = (auc2*(1-auc2) + (n_pos-1)*(q1b-auc2**2) + (n_neg-1)*(q2b-auc2**2)) / (n_pos*n_neg)
    # Correlation r ~ 0.75 for same dataset different methods (conservative)
    r = 0.75
    se_diff = np.sqrt(var1 + var2 - 2*r*np.sqrt(var1*var2))
    z = (auc1 - auc2) / se_diff
    p = 2 * (1 - norm.cdf(abs(z)))
    return p

# Sample sizes from clinical data
n_pos, n_neg = high_risk_n, low_risk_n
print(f"Sample sizes: n_pos={n_pos}, n_neg={n_neg}")

# Mean-Z vs ssGSEA AUCs from Table 10 (Mean-Z) and Table 11 (ssGSEA)
comparisons = [
    ('Elastic Net', 0.641, 0.631),
    ('Random Forest', 0.645, 0.622),
    ('Gradient Boosting', 0.633, 0.595),
]

for name, auc_z, auc_ssgsea in comparisons:
    p = delong_test(auc_z, auc_ssgsea, n_pos, n_neg)
    print(f"{name}: delta={auc_z-auc_ssgsea:.3f}, p={p:.3f}")

### Section 4: Hazard ratios from combined Cox model

In [ ]:
# Load combined features for GSE96058
# Requires clinical features and pathway scores
if os.path.exists(gse_scores_path):
    pathway_scores = pd.read_csv(gse_scores_path).set_index('sample_id')
    gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
    clinical_features = encode_clinical_features(gse_clin).set_index(gse_clin['sample_id'])
    
    # Align and build combined matrix
    common_idx = pathway_scores.index.intersection(clinical_features.index)
    X_comb = build_feature_matrix(pathway_scores.loc[common_idx], clinical_features.loc[common_idx])
    
    # Survival data
    gse_clin = gse_clin.set_index('sample_id').loc[common_idx]
    X_comb['time_to_event'] = gse_clin['time_to_event']
    X_comb['event_status'] = gse_clin['event_status']
    
    # Fit Cox on full data (not CV) to get HRs for reporting
    print("\nFitting Cox model on combined features...")
    cox = CoxPHFitter(penalizer=0.1)
    cox.fit(X_comb, duration_col='time_to_event', event_col='event_status')
    cox.print_summary()
    
    hr_df = cox.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
    hr_df.columns = ['HR', 'CI_lower', 'CI_upper', 'p_value']
    hr_df.to_csv('../results/cox_hazard_ratios.csv')
    print("\nSaved HRs to results/cox_hazard_ratios.csv")
    print(hr_df)
else:
    print(f"Warning: {gse_scores_path} not found. Skipping Section 4.")